# STEP 02 — species-agnostic loader + smoke-test EXP047-B pipeline

Цель этого notebook — **не обучать новую модель** и не менять рецепт EXP047-B.

Здесь делаем только инфраструктурный шаг:

1. безопасно патчим `src/ml_project/epic_data.py`;
2. разрешаем species-specific filtered `csRNA-*-r1/r2-train.bed.gz`;
3. корректно читаем целые counts, записанные в scientific notation;
4. проверяем, что Nematostella продолжает загружаться без изменения старого split;
5. создаём development split для первого вида `ilPloInte3.2`;
6. собираем один CPU batch той же геометрии, которую использует EXP047-B.

**Никакого training, test-label access или leaderboard tuning в этом notebook нет.**


In [1]:
from pathlib import Path
import gc
import hashlib
import importlib
import json
import sys
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display

EXPECTED_PYTHON = Path(r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe").resolve()
ACTUAL_PYTHON = Path(sys.executable).resolve()
assert ACTUAL_PYTHON == EXPECTED_PYTHON, (ACTUAL_PYTHON, EXPECTED_PYTHON)

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (current, *current.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src" / "ml_project").is_dir()
    and (candidate / "data" / "raw" / "unpacked").is_dir()
)

AGENTS_PATH = next(
    (
        path
        for path in (
            PROJECT_ROOT / "AGENTS.md",
            PROJECT_ROOT.parent / "AGENTS.md",
        )
        if path.is_file()
    ),
    None,
)
assert AGENTS_PATH is not None, "AGENTS.md not found"
agents_bytes = AGENTS_PATH.read_bytes()
agents_sha = hashlib.sha256(agents_bytes).hexdigest()
EXPECTED_AGENTS_SHA = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
assert agents_sha == EXPECTED_AGENTS_SHA, (
    "AGENTS.md changed since the audited workflow. "
    f"Current={agents_sha}, expected={EXPECTED_AGENTS_SHA}. "
    "Stop and review the new instructions before continuing."
)

print("Python:", ACTUAL_PYTHON)
print("Project:", PROJECT_ROOT)
print("AGENTS:", AGENTS_PATH)
print("AGENTS SHA256:", agents_sha)
print()
print(agents_bytes.decode("utf-8"))


Python: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
Project: D:\Projects\EPIC\EPIC
AGENTS: D:\Projects\EPIC\AGENTS.md
AGENTS SHA256: bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3

# Project agent instructions

- Run all Python commands, notebooks, analyses, and tests for this workspace in the Conda environment `epic-eda`.
- On this Windows machine, prefer the explicit interpreter `C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe` when invoking Python non-interactively.
- Do not use the repository `.venv` or the system Python for project work unless the user explicitly requests it.

## Decision-making and scientific judgment

- Do not automatically agree with the user's proposed interpretation, experiment, architecture, or next step.
- Treat the user's preferred direction as a hypothesis to evaluate, not as the conclusion to defend.
- Independently compare the available alternatives using experimental evidence, expected information gain, methodolo

## 1. Безопасный patch `epic_data.py`

Patch применяется только если SHA-256 файла совпадает либо с присланной тобой исходной версией,
либо с уже ожидаемой patched-версией. Любое третье состояние останавливает notebook.

Исходный файл сохраняется в `artifacts/patches/EPIC_5SPECIES_LOADER/`.


In [2]:
from pathlib import Path
import hashlib

SOURCE_PATH = PROJECT_ROOT / "src" / "ml_project" / "epic_data.py"
PATCH_DIR = PROJECT_ROOT / "artifacts" / "patches" / "EPIC_5SPECIES_LOADER"
PATCH_DIR.mkdir(parents=True, exist_ok=True)

ORIGINAL_SHA = "6be389ecf29ebb10232c7d0146e99fbe047690e830647a79567de67f7b1d1547"
PATCHED_SHA = "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468"

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

current_bytes = SOURCE_PATH.read_bytes()
current_sha = sha256_bytes(current_bytes)
print("epic_data.py current SHA256:", current_sha)

if current_sha == PATCHED_SHA:
    print("Patch already applied; no source rewrite needed.")

elif current_sha == ORIGINAL_SHA:
    backup = PATCH_DIR / f"epic_data.py.before_species_patch.{ORIGINAL_SHA}.bak"
    if backup.exists():
        assert backup.read_bytes() == current_bytes, "Existing backup differs from current original source"
    else:
        backup.write_bytes(current_bytes)

    source = current_bytes.decode("utf-8")

    old_import = """from dataclasses import dataclass
import gzip
"""
    new_import = """from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
import gzip
"""
    assert old_import in source
    source = source.replace(old_import, new_import, 1)

    marker = """def paths_for(
    project_root: Path | str,
    config: SplitConfig = DEFAULT_SPLIT_CONFIG,
) -> EpicPaths:
"""
    helper = """def _discover_filtered_replicate_pair(
    participants: Path,
) -> tuple[Path, Path]:
    # Найти единственную filtered пару csRNA r1/r2 для assembly.
    r1_candidates = sorted(
        path
        for path in participants.glob("csRNA*-r1-train.bed*")
        if "unfiltered" not in path.name.lower()
    )
    r2_candidates = sorted(
        path
        for path in participants.glob("csRNA*-r2-train.bed*")
        if "unfiltered" not in path.name.lower()
    )
    r2_by_key = {
        path.name.replace("-r2-train", "-rX-train"): path
        for path in r2_candidates
    }
    pairs = []
    for r1_path in r1_candidates:
        key = r1_path.name.replace("-r1-train", "-rX-train")
        r2_path = r2_by_key.get(key)
        if r2_path is not None:
            pairs.append((r1_path, r2_path))

    if len(pairs) != 1:
        details = ", ".join(
            f"{first.name} + {second.name}" for first, second in pairs
        ) or "no matched filtered pairs"
        raise FileNotFoundError(
            "Ожидалась ровно одна filtered csRNA r1/r2 пара в "
            f"{participants}, найдено {len(pairs)}: {details}"
        )
    return pairs[0]


def paths_for(
    project_root: Path | str,
    config: SplitConfig = DEFAULT_SPLIT_CONFIG,
) -> EpicPaths:
"""
    assert marker in source
    source = source.replace(marker, helper, 1)

    old_paths = """    inputs = {
        "fai": source / "genome/genome.fa.fai",
        "whitelist_train": participants / "whitelist.train.bed.gz",
        "whitelist_test": participants / "whitelist.test.bed.gz",
        "template_train": participants / "template.train.bed.gz",
        "template_test": participants / "template.test.bed.gz",
        "csrna_r1": participants / "csRNA-r1-train.bed.gz",
        "csrna_r2": participants / "csRNA-r2-train.bed.gz",
    }
"""
    new_paths = """    csrna_r1, csrna_r2 = _discover_filtered_replicate_pair(participants)
    inputs = {
        "fai": source / "genome/genome.fa.fai",
        "whitelist_train": participants / "whitelist.train.bed.gz",
        "whitelist_test": participants / "whitelist.test.bed.gz",
        "template_train": participants / "template.train.bed.gz",
        "template_test": participants / "template.test.bed.gz",
        "csrna_r1": csrna_r1,
        "csrna_r2": csrna_r2,
    }
"""
    assert old_paths in source
    source = source.replace(old_paths, new_paths, 1)

    old_count = """            contig, start, end, _name, count, strand = fields
            start, end, count = int(start), int(end), int(count)
            if end <= start or count <= 0 or strand not in {"+", "-"}:
                raise ValueError("Неверный сигнальный BED-интервал")
"""
    new_count = """            contig, start, end, _name, count_text, strand = fields
            start, end = int(start), int(end)
            try:
                count_decimal = Decimal(count_text)
            except InvalidOperation as error:
                raise ValueError(
                    f"Неверный csRNA count: {count_text!r}"
                ) from error
            if (
                not count_decimal.is_finite()
                or count_decimal <= 0
                or count_decimal != count_decimal.to_integral_value()
            ):
                raise ValueError(
                    f"csRNA count должен быть положительным целым: {count_text!r}"
                )
            count = int(count_decimal)
            if end <= start or strand not in {"+", "-"}:
                raise ValueError("Неверный сигнальный BED-интервал")
"""
    assert old_count in source
    source = source.replace(old_count, new_count, 1)

    compile(source, str(SOURCE_PATH), "exec")
    patched_bytes = source.encode("utf-8")
    actual_patched_sha = sha256_bytes(patched_bytes)
    assert actual_patched_sha == PATCHED_SHA, (actual_patched_sha, PATCHED_SHA)

    SOURCE_PATH.write_bytes(patched_bytes)
    assert sha256_bytes(SOURCE_PATH.read_bytes()) == PATCHED_SHA

    patch_manifest = {
        "source": str(SOURCE_PATH),
        "original_sha256": ORIGINAL_SHA,
        "patched_sha256": PATCHED_SHA,
        "changes": [
            "discover exactly one filtered species-specific csRNA r1/r2 pair",
            "exclude filenames containing 'unfiltered'",
            "parse integer counts exactly through Decimal, including scientific notation",
        ],
        "backup": str(backup),
    }
    (PATCH_DIR / "patch_manifest.json").write_text(
        json.dumps(patch_manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("Patch applied:", SOURCE_PATH)
    print("Backup:", backup)

else:
    raise RuntimeError(
        "epic_data.py differs from both the audited original and expected patched version. "
        f"Current SHA256={current_sha}. Stop: do not patch an unknown source."
    )


epic_data.py current SHA256: 6be389ecf29ebb10232c7d0146e99fbe047690e830647a79567de67f7b1d1547
Patch applied: D:\Projects\EPIC\EPIC\src\ml_project\epic_data.py
Backup: D:\Projects\EPIC\EPIC\artifacts\patches\EPIC_5SPECIES_LOADER\epic_data.py.before_species_patch.6be389ecf29ebb10232c7d0146e99fbe047690e830647a79567de67f7b1d1547.bak


## 2. Перезагрузить модуль и выполнить два локальных unit smoke-test

Здесь не читаются большие геномы.

Проверяем отдельно:

- species-specific pair discovery и игнорирование `unfiltered`;
- `1.43057e+06` читается как точный integer `1430570`, а не через опасное округление float.


In [3]:
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import ml_project.epic_data as epic_data
epic_data = importlib.reload(epic_data)

with tempfile.TemporaryDirectory() as tmp:
    participants = Path(tmp)
    for name in (
        "csRNA-Embryo-r1-train.bed.gz",
        "csRNA-Embryo-r2-train.bed.gz",
        "csRNA-Embryo-unfiltered-r1-train.bed.gz",
        "csRNA-Embryo-unfiltered-r2-train.bed.gz",
    ):
        (participants / name).touch()

    r1, r2 = epic_data._discover_filtered_replicate_pair(participants)
    assert r1.name == "csRNA-Embryo-r1-train.bed.gz"
    assert r2.name == "csRNA-Embryo-r2-train.bed.gz"

import gzip

with tempfile.TemporaryDirectory() as tmp:
    signal_path = Path(tmp) / "signal.bed.gz"
    with gzip.open(signal_path, "wt", encoding="ascii") as stream:
        stream.write("chr1\t10\t11\t.\t1.43057e+06\t+\n")

    whitelist = pd.DataFrame(
        {
            "contig": ["chr1"],
            "start": np.array([0], dtype=np.int64),
            "end": np.array([20], dtype=np.int64),
        }
    )
    signal = epic_data.read_sparse_signal(signal_path, whitelist, "r1_count")
    assert len(signal) == 1
    assert int(signal.iloc[0]["r1_count"]) == 1_430_570
    assert int(signal.iloc[0]["coordinate_0based"]) == 10
    assert signal.iloc[0]["strand"] == "+"

print("Synthetic loader tests: PASS")


Synthetic loader tests: PASS


## 3. Разрешение файлов для всех 6 assemblies

Это только проверка путей. Split для пяти конкурсных видов здесь ещё не строится.


In [4]:
ASSEMBLIES = [
    "jaNemVect1.1",
    "ilPloInte3.2",
    "cgigas_uk_roslin_v1",
    "Ofas_2.0",
    "ASM119413v2",
    "sSquSuc5.hap2",
]

rows = []
for assembly in ASSEMBLIES:
    config = epic_data.SplitConfig(assembly=assembly)
    paths = epic_data.paths_for(PROJECT_ROOT, config)
    missing = [key for key, path in paths.inputs.items() if not path.is_file()]
    assert not missing, (assembly, missing)
    assert "unfiltered" not in paths.inputs["csrna_r1"].name.lower()
    assert "unfiltered" not in paths.inputs["csrna_r2"].name.lower()
    rows.append(
        {
            "assembly": assembly,
            "r1": paths.inputs["csrna_r1"].name,
            "r2": paths.inputs["csrna_r2"].name,
            "all_required_files_exist": True,
        }
    )

resolved = pd.DataFrame(rows)
display(resolved)

nem_row = resolved.loc[resolved["assembly"] == "jaNemVect1.1"].iloc[0]
assert nem_row["r1"] == "csRNA-r1-train.bed.gz"
assert nem_row["r2"] == "csRNA-r2-train.bed.gz"

print("All 6 assemblies resolved: PASS")


,assembly,r1,r2,all_required_files_exist
0,jaNemVect1.1,csRNA-r1-train.bed.gz,csRNA-r2-train.bed.gz,True
1,ilPloInte3.2,csRNA-Embryo-r1-train.bed.gz,csRNA-Embryo-r2-train.bed.gz,True
2,cgigas_uk_roslin_v1,csRNA-Heart-r1-train.bed.gz,csRNA-Heart-r2-train.bed.gz,True
3,Ofas_2.0,csRNA-Embryo-r1-train.bed.gz,csRNA-Embryo-r2-train.bed.gz,True
4,ASM119413v2,csRNA-Kidney-r1-train.bed.gz,csRNA-Kidney-r2-train.bed.gz,True
5,sSquSuc5.hap2,csRNA-Blood-r1-train.bed.gz,csRNA-Blood-r2-train.bed.gz,True


All 6 assemblies resolved: PASS


## 4. Regression test — старый Nematostella split

Здесь **не пересобираем** исторический split. Загружаем уже существующий artifact с проверкой SHA,
чтобы доказать, что loader patch не изменил старый data contract.


In [5]:
NEM_CONFIG = epic_data.SplitConfig(
    assembly="jaNemVect1.1",
    split_version="contig_holdout_v1",
    seed=42,
    validation_contig_fraction=0.25,
)

nem = epic_data.load_split(PROJECT_ROOT, NEM_CONFIG, verify_hashes=True)
nem_summary = nem.summary.copy()
display(nem_summary)

trainval_positions = int(
    nem_summary.loc[
        nem_summary["split"].isin(["train", "validation"]),
        "position_strand_count",
    ].sum()
)
test_positions = int(
    nem_summary.loc[nem_summary["split"] == "test", "position_strand_count"].iloc[0]
)
trainval_positives = int(
    nem_summary.loc[
        nem_summary["split"].isin(["train", "validation"]),
        "positive_positions",
    ].sum()
)

assert trainval_positions == 311_871_646
assert test_positions == 72_613_390
assert trainval_positives == 226_648
assert nem.paths.inputs["csrna_r1"].name == "csRNA-r1-train.bed.gz"
assert nem.paths.inputs["csrna_r2"].name == "csRNA-r2-train.bed.gz"

print("Nematostella regression: PASS")


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Nematostella regression: PASS


## 5. Подготовить первый species split — `ilPloInte3.2`

Теперь строим тот же `contig_holdout_v1`, но уже для моли.

Это development split для воспроизведения EXP045 → EXP047-B.
Official test остаётся без labels и не используется в training/mining.


In [6]:
MOTH_ASSEMBLY = "ilPloInte3.2"
MOTH_CONFIG = epic_data.SplitConfig(
    assembly=MOTH_ASSEMBLY,
    split_version="contig_holdout_v1",
    seed=42,
    validation_contig_fraction=0.25,
)

moth = epic_data.ensure_prepared_split(PROJECT_ROOT, MOTH_CONFIG)
display(moth.summary)
display(moth.contig_assignment)

assert set(moth.contig_assignment["split"]) == {"train", "validation", "test"}
assert set(moth.intervals("train")["contig"]).isdisjoint(
    set(moth.intervals("validation")["contig"])
)
assert set(moth.intervals("train")["contig"]).isdisjoint(
    set(moth.intervals("test")["contig"])
)
assert set(moth.intervals("validation")["contig"]).isdisjoint(
    set(moth.intervals("test")["contig"])
)
assert moth.positive_targets("test") is None

print("Moth split prepared:", moth.paths.output)


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,19,178622,164558039,329116078,60.145242,581052.0,328535026.0,0.176549
1,validation,7,63150,52057100,104114200,19.026642,162306.0,103951894.0,0.155892
2,test,7,67434,56985954,113971908,20.828116,NaN,NaN,NaN


,contig,split,source_split
0,NC_071294.1,train,train
1,NC_071295.1,train,train
2,NC_071296.1,train,train
3,NC_071297.1,train,train
4,NC_071298.1,train,train
5,NC_071299.1,validation,train
6,NC_071300.1,train,train
7,NC_071301.1,validation,train
8,NC_071304.1,train,train
9,NC_071305.1,train,train


Moth split prepared: D:\Projects\EPIC\EPIC\data\processed\ilPloInte3.2\splits\contig_holdout_v1


## 6. EXP047-B geometry smoke-test на моли

Загружаем FASTA **только для первого вида**, собираем один CPU batch и проверяем совместимость
с неизменённой WIDTH512 architecture.

GPU/DirectML здесь не используется и optimizer не создаётся.


In [7]:
import torch

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
)

sequences, fasta_alphabet = load_baseline_sequences(moth)

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)
generator = ProfileWindowGenerator(
    prepared_split=moth,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
batch = generator.sample_batch(batch_size=4)

assert tuple(batch.x.shape) == (4, 5, 1284)
assert tuple(batch.x_both_strands.shape) == (8, 5, 1284)
assert tuple(batch.target.shape) == (4, 2, 1024)
assert tuple(batch.count.shape) == (4, 2, 1024)
assert tuple(batch.mask.shape) == (4, 2, 1024)
assert torch.all(batch.target.bool() <= batch.mask)

model = create_cnn_presence_intensity(channels=512)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
assert parameter_count == 14_708_738
assert model.backbone.stem.in_channels == 5
assert model.backbone.stem.out_channels == 512

print("Batch x:", tuple(batch.x.shape))
print("Batch target:", tuple(batch.target.shape))
print("Batch valid masked position-strands:", int(batch.mask.sum()))
print("Batch positive position-strands:", int(batch.target.sum()))
print("WIDTH512 parameters:", f"{parameter_count:,}")
print("Moth EXP047-B geometry smoke-test: PASS")


Batch x: (4, 5, 1284)
Batch target: (4, 2, 1024)
Batch valid masked position-strands: 7912
Batch positive position-strands: 67
WIDTH512 parameters: 14,708,738
Moth EXP047-B geometry smoke-test: PASS


## 7. Зафиксировать результат шага

После `PASS` следующая работа — отдельный notebook для **первой training stage моли:
EXP045-equivalent WIDTH512 @50k**. Рецепт EXP047-B пока не меняем.


In [8]:
REPORT_DIR = PROJECT_ROOT / "artifacts" / "audits" / "EPIC_MOTH_EXP047B_READINESS"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

source_sha = hashlib.sha256(
    (PROJECT_ROOT / "src/ml_project/epic_data.py").read_bytes()
).hexdigest()

report = {
    "assembly": MOTH_ASSEMBLY,
    "loader_patch_sha256": source_sha,
    "expected_loader_patch_sha256": "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468",
    "split_version": MOTH_CONFIG.split_version,
    "seed": MOTH_CONFIG.seed,
    "validation_contig_fraction": MOTH_CONFIG.validation_contig_fraction,
    "resolved_inputs": {
        key: str(path.relative_to(PROJECT_ROOT))
        for key, path in moth.paths.inputs.items()
    },
    "summary": moth.summary.to_dict(orient="records"),
    "batch_shapes": {
        "x": list(batch.x.shape),
        "x_both_strands": list(batch.x_both_strands.shape),
        "target": list(batch.target.shape),
        "count": list(batch.count.shape),
        "mask": list(batch.mask.shape),
    },
    "width512_parameter_count": parameter_count,
    "status": "PASS",
    "next_stage": "EXP045-equivalent WIDTH512 50k for ilPloInte3.2",
}

report_path = REPORT_DIR / "readiness_report.json"
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("Report:", report_path)
print(json.dumps(report, ensure_ascii=False, indent=2))

del model, batch, generator, sequences, fasta_alphabet
gc.collect()


Report: D:\Projects\EPIC\EPIC\artifacts\audits\EPIC_MOTH_EXP047B_READINESS\readiness_report.json
{
  "assembly": "ilPloInte3.2",
  "loader_patch_sha256": "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468",
  "expected_loader_patch_sha256": "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468",
  "split_version": "contig_holdout_v1",
  "seed": 42,
  "validation_contig_fraction": 0.25,
  "resolved_inputs": {
    "fai": "data\\raw\\unpacked\\ilPloInte3.2\\genome\\genome.fa.fai",
    "whitelist_train": "data\\raw\\unpacked\\ilPloInte3.2\\for_participants\\whitelist.train.bed.gz",
    "whitelist_test": "data\\raw\\unpacked\\ilPloInte3.2\\for_participants\\whitelist.test.bed.gz",
    "template_train": "data\\raw\\unpacked\\ilPloInte3.2\\for_participants\\template.train.bed.gz",
    "template_test": "data\\raw\\unpacked\\ilPloInte3.2\\for_participants\\template.test.bed.gz",
    "csrna_r1": "data\\raw\\unpacked\\ilPloInte3.2\\for_participants\\csRNA-Embryo-r1-trai

96